Setup & Load Data Bersih

In [1]:
# Menghubungkan Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Load data yang sudah dibersihkan di Notebook 1
file_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/data/cleaned_enron_data.csv'
df = pd.read_csv(file_path)

# PENGAMAN: Hapus baris jika ada nilai NaN (kosong) yang terbuat otomatis saat read_csv
df = df.dropna(subset=['text'])

# Mengecek apakah data bersih berhasil dimuat
df.head()

,text,label
0,christmas tree farm pictures,ham
1,vastar resources inc gary production high isla...,ham
2,calpine daily gas nomination calpine daily gas...,ham
3,issue fyi see note already done stella forward...,ham
4,meter nov allocation fyi forwarded lauri allen...,ham


Memisahkan Fitur (X) dan Target (y)

In [3]:
# X adalah kolom teks, y adalah kolom label
X = df['text']
y = df['label']

print("Jumlah data X:", len(X))
print("Jumlah data y:", len(y))

Jumlah data X: 33716
Jumlah data y: 33716


### 1. Data Splitting (Pembagian Data)
Membagi dataset menjadi Data Latih (80%) dan Data Uji (20%) menggunakan pustaka `scikit-learn`.

In [4]:
from sklearn.model_selection import train_test_split

# Membagi data dengan rasio 80:20
# random_state=42 digunakan agar pengacakan datanya konsisten setiap kali kode di-run
# Menambahkan stratify=y agar pembagian label Spam dan Ham seimbang
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Cek distribusi soal ujian yang baru
print("Distribusi label di Data Uji:")
print(y_test.value_counts())

print("\nJumlah Data Latih (Training):", len(X_train))
print("Jumlah Data Uji (Testing):", len(X_test))

Distribusi label di Data Uji:
label
spam    3435
ham     3309
Name: count, dtype: int64

Jumlah Data Latih (Training): 26972
Jumlah Data Uji (Testing): 6744


In [5]:
# Mengecek total populasi Spam dan Ham di seluruh data
print("Total keseluruhan distribusi label:")
print(y.value_counts())

Total keseluruhan distribusi label:
label
spam    17171
ham     16545
Name: count, dtype: int64


### 2. Feature Extraction (Vektorisasi TF-IDF)
Mengubah teks (X_train dan X_test) menjadi representasi matriks angka.
Catatan: Kita HANYA melatih TF-IDF (`fit`) pada Data Latih untuk mencegah *data leakage*, lalu menerapkannya (`transform`) ke Data Latih dan Data Uji.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi fungsi TF-IDF (maksimal mengambil 5000 kata paling penting agar tidak berat)
tfidf = TfidfVectorizer()

# 1. Mesin belajar dari kata-kata di Data Latih (fit), lalu mengubahnya jadi angka (transform)
X_train_tfidf = tfidf.fit_transform(X_train)

# 2. Mesin mengubah Data Uji menjadi angka berdasarkan apa yang ia pelajari dari Data Latih
X_test_tfidf = tfidf.transform(X_test)

print("Bentuk matriks Data Latih:", X_train_tfidf.shape)
print("Bentuk matriks Data Uji:", X_test_tfidf.shape)

Bentuk matriks Data Latih: (26972, 46065)
Bentuk matriks Data Uji: (6744, 46065)


### 3. Pembangunan dan Pelatihan Model (Bonus +10)
Melatih dan membandingkan performa 2 algoritma Machine Learning:
1. Naive Bayes (MultinomialNB)
2. Support Vector Machine (LinearSVC)

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
import time

# 1. Inisialisasi Model
nb_model = MultinomialNB()
svm_model = LinearSVC(random_state=42)

# 2. Melatih Model Naive Bayes
print("Sedang melatih Naive Bayes...")
start_time = time.time()
nb_model.fit(X_train_tfidf, y_train)
print(f"Selesai! (Waktu: {time.time() - start_time:.2f} detik)\n")

# 3. Melatih Model SVM
print("Sedang melatih SVM...")
start_time = time.time()
svm_model.fit(X_train_tfidf, y_train)
print(f"Selesai! (Waktu: {time.time() - start_time:.2f} detik)")

Sedang melatih Naive Bayes...
Selesai! (Waktu: 0.16 detik)

Sedang melatih SVM...
Selesai! (Waktu: 1.50 detik)


### 4. Evaluasi Model
Menguji model yang sudah pintar menggunakan Data Uji (X_test_tfidf) dan membandingkan tebakannya dengan kunci jawaban asli (y_test).

In [8]:
from sklearn.metrics import accuracy_score, classification_report

# Menyuruh model menebak soal ujian
nb_predictions = nb_model.predict(X_test_tfidf)
svm_predictions = svm_model.predict(X_test_tfidf)

# Menghitung akurasi (Berapa persen tebakannya yang benar?)
nb_accuracy = accuracy_score(y_test, nb_predictions)
svm_accuracy = accuracy_score(y_test, svm_predictions)

print(f"Akurasi Naive Bayes: {nb_accuracy * 100:.2f}%")
print(f"Akurasi SVM        : {svm_accuracy * 100:.2f}%\n")

print("-" * 50)
print("Detail Performa NB:")
print(classification_report(y_test, nb_predictions))

print("-" * 50)
print("Detail Performa SVM:")
print(classification_report(y_test, svm_predictions))

Akurasi Naive Bayes: 98.22%
Akurasi SVM        : 99.94%

--------------------------------------------------
Detail Performa NB:
              precision    recall  f1-score   support

         ham       1.00      0.96      0.98      3309
        spam       0.97      1.00      0.98      3435

    accuracy                           0.98      6744
   macro avg       0.98      0.98      0.98      6744
weighted avg       0.98      0.98      0.98      6744

--------------------------------------------------
Detail Performa SVM:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00      3309
        spam       1.00      1.00      1.00      3435

    accuracy                           1.00      6744
   macro avg       1.00      1.00      1.00      6744
weighted avg       1.00      1.00      1.00      6744



In [10]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def bersihkan_teks_uji(teks):
    teks = str(teks).lower()
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    kata_bersih = [kata for kata in teks.split() if kata not in stop_words]
    return ' '.join(kata_bersih)

# 1. Kita masukkan dua teks (satu spam modern, satu spam jadul Enron)
teks_phishing_paypal = "URGENT: Your PayPal account has been suspended due to suspicious activity. Please click the link below to verify your identity and restore your access immediately. Failure to do so will result in permanent account deletion."

teks_spam_jadul = "Buy cheap viagra and cialis pills online without prescription! 100% guaranteed weight loss pharmacy deals. Click here for hot stock investment and win million dollars today!"

# 2. Bersihkan teks
teks_1_bersih = bersihkan_teks_uji(teks_phishing_paypal)
teks_2_bersih = bersihkan_teks_uji(teks_spam_jadul)

# 3. Ubah jadi angka pakai TF-IDF yang baru saja di-training
vektor_uji = tfidf.transform([teks_1_bersih, teks_2_bersih])

# 4. Suruh SVM menebak
prediksi = svm_model.predict(vektor_uji)

print("--- HASIL UJI ISOLASI ---")
print("Tebakan PayPal Phishing :", prediksi[0])
print("Tebakan Spam Viagra     :", prediksi[1])
print("\nKategori yang dikenal SVM:", svm_model.classes_)

--- HASIL UJI ISOLASI ---
Tebakan PayPal Phishing : ham
Tebakan Spam Viagra     : ham

Kategori yang dikenal SVM: ['ham' 'spam']


### 5. Export Model & Vectorizer
Menyimpan model terbaik dan TF-IDF Vectorizer menggunakan `joblib` agar bisa digunakan di aplikasi Streamlit.

In [9]:
import joblib

# Tentukan jalur penyimpanan ke folder 'models' di Drive
model_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/models/spam_model_svm.pkl'
tfidf_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/models/tfidf_vectorizer.pkl'

# Menyimpan Model SVM
joblib.dump(svm_model, model_path)

# Menyimpan fungsi TF-IDF (SANGAT PENTING!)
joblib.dump(tfidf, tfidf_path)

print("Model dan TF-IDF Vectorizer berhasil disimpan!")
print(f"Lokasi Model : {model_path}")
print(f"Lokasi TF-IDF: {tfidf_path}")

Model dan TF-IDF Vectorizer berhasil disimpan!
Lokasi Model : /content/drive/MyDrive/Semester 4/Machine Learning/Final Project/models/spam_model_svm.pkl
Lokasi TF-IDF: /content/drive/MyDrive/Semester 4/Machine Learning/Final Project/models/tfidf_vectorizer.pkl


In [12]:
import pandas as pd
import numpy as np

# 1. Ambil seluruh daftar kata yang diketahui oleh TF-IDF
kamus_kata = np.array(tfidf.get_feature_names_out())

# 2. Ambil nilai bobot matematis dari otak SVM
# (Bobot positif = Mesin menganggapnya Spam, Bobot negatif = Ham)
bobot_svm = svm_model.coef_[0]

# 3. Gabungkan ke dalam tabel agar mudah dibaca
df_otak = pd.DataFrame({'Kata': kamus_kata, 'Bobot_Spam': bobot_svm})

# 4. Tampilkan 10 Kata Paling "Spam" dan Paling "Ham" di mata mesin
top_10_spam = df_otak.sort_values(by='Bobot_Spam', ascending=False)
# top_10_ham = df_otak.sort_values(by='Bobot_Spam', ascending=True).head(10)

# print("🚨 TOP 10 KATA YANG MENURUT MESIN ADALAH SPAM 🚨")
print(top_10_spam)

# print("\n✅ TOP 10 KATA YANG MENURUT MESIN ADALAH HAM ✅")
# print(top_10_ham)

           Kata  Bobot_Spam
42045     trdts    2.026309
27304     nahou    2.024082
42563       ubs    1.622320
558       admin    1.528329
24440   machine    1.527253
...         ...         ...
44482       web   -0.523598
12453   ectstca   -0.533478
32189  progress   -0.565341
12446    ecthou   -1.268200
4661        bps   -1.291029

[46065 rows x 2 columns]
